# Лабораторная работа №8

## Скрапинг и анализ текста

**Выполнил:** Корнеев Фёдор, группа P3120

Цель работы — собрать новости с сайта ITMO.NEWS, сохранить общую информацию о публикациях, а затем получить подробные данные для каждой новости: заголовок, дату, количество просмотров, текст и теги.

## 1. Импорт библиотек

Для загрузки страниц используется `requests`, для разбора HTML — `BeautifulSoup`, а для хранения и сохранения результатов — `pandas`.

Во время тестирования используется ограниченный набор страниц. После проверки парсера тестовый режим будет отключён.

In [1]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE_URL = "https://news.itmo.ru"
MAIN_NEWS_URL = BASE_URL + "/ru/main_news/{page}/"

OUTPUT_DIR = Path("news_content")
OUTPUT_DIR.mkdir(exist_ok=True)

GENERAL_CSV = Path("news.csv")
CONTENT_CSV = OUTPUT_DIR / "news_content.csv"

# Этап проверки:
# полный индекс новостей собираем со всех страниц,
# но содержимое пока проверяем только на 3 статьях.
FULL_LISTING = True
FULL_CONTENT = True

TEST_PAGES = 2
TEST_ARTICLES = 10

REQUEST_DELAY = 0.25
REQUEST_TIMEOUT = 30

print("Полный список новостей:", FULL_LISTING)
print("Полный парсинг содержимого:", FULL_CONTENT)
print("Общий CSV:", GENERAL_CSV)
print("CSV с содержимым:", CONTENT_CSV)


Полный список новостей: True
Полный парсинг содержимого: True
Общий CSV: news.csv
CSV с содержимым: news_content/news_content.csv


## 2. HTTP-сессия и вспомогательные функции

Используется одна `requests.Session`, чтобы не создавать новое соединение для каждого запроса. Также задаётся обычный браузерный `User-Agent`.

In [2]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "Chrome/140.0 Safari/537.36"
    )
})

retry = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)


def clean_text(value):
    """Удаляет лишние пробелы из текста."""
    if value is None:
        return None

    return " ".join(str(value).split())


def get_soup(url):
    """Загружает HTML-страницу и возвращает BeautifulSoup."""
    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")


## 3. Парсинг страницы со списком новостей

Для каждой новости из общего списка собираются:

- идентификатор новости;
- название;
- дата размещения;
- URL.

Идентификатор извлекается из URL вида `/news/15033/`.

In [3]:
NEWS_ID_RE = re.compile(r"/news/(\d+)/?")


def parse_listing_page(page_number):
    url = MAIN_NEWS_URL.format(page=page_number)
    soup = get_soup(url)

    result = []

    for item in soup.select("ul.triplet > li"):
        # У карточки новости может быть несколько ссылок:
        # картинка с пустым текстом и отдельная ссылка-заголовок.
        news_links = []

        for link in item.find_all("a", href=True):
            href = link.get("href", "")

            if NEWS_ID_RE.search(href):
                news_links.append(link)

        if not news_links:
            continue

        link = news_links[0]
        href = link.get("href", "")

        match = NEWS_ID_RE.search(href)

        if match is None:
            continue

        news_id = int(match.group(1))

        # Сначала пытаемся найти непустую текстовую ссылку
        # на ту же новость.
        title = None

        for candidate in news_links:
            candidate_text = clean_text(
                candidate.get_text(" ", strip=True)
            )

            if candidate_text:
                title = candidate_text
                break

        # Резервный вариант на случай другой верстки.
        if not title:
            heading = item.find(["h2", "h3", "h4"])

            if heading is not None:
                title = clean_text(
                    heading.get_text(" ", strip=True)
                )

        if not title:
            continue

        time_tag = item.find("time")

        if time_tag is None:
            continue

        date = (
            time_tag.get("datetime")
            or clean_text(time_tag.get_text(" ", strip=True))
        )

        news_url = urljoin(BASE_URL, href)

        result.append({
            "id": news_id,
            "title": title,
            "date": date,
            "url": news_url,
        })

    # На всякий случай исключаем повтор одной и той же новости,
    # если HTML содержит несколько одинаковых карточек.
    unique = {}

    for row in result:
        unique[row["id"]] = row

    result = list(unique.values())

    next_link = None

    for link in soup.select("div.pagination a"):
        if clean_text(link.get_text()) == "Следующая":
            next_link = link
            break

    has_next = bool(
        next_link
        and next_link.get("href")
        and next_link.get("href") != "#"
        and "disabled" not in next_link.get("class", [])
    )

    return result, has_next


test_rows, test_has_next = parse_listing_page(1)

print("Новостей на первой странице:", len(test_rows))
print("Есть следующая страница:", test_has_next)

pd.DataFrame(test_rows)


Новостей на первой странице: 9
Есть следующая страница: True


,id,title,date,url
0,15033,Соединили пышки и квантовую механику: как ИТМО...,2026-09-21T18:24:25,https://news.itmo.ru/ru/education/cooperation/...
1,15018,Эра сверхбыстрых вычислений: в ИТМО появился И...,2026-09-15T16:40:38,https://news.itmo.ru/ru/science/photonics/news...
2,15003,Олимпиадный год: итоги самой масштабной приемн...,2026-09-01T19:18:10,https://news.itmo.ru/ru/education/official/new...
3,15001,ITMO CONF 2026: как идеи проходят путь от лабо...,2026-08-31T18:44:11,https://news.itmo.ru/ru/education/trend/news/1...
4,14985,Всем по киберпышке! ИТМО и Яндекс Образование ...,2026-08-18T15:57:07,https://news.itmo.ru/ru/education/cooperation/...
5,14984,Российские школьники взяли четыре медали на Ме...,2026-08-17T16:39:54,https://news.itmo.ru/ru/university_live/achiev...
6,14973,В ИТМО завершили прием на бюджет бакалавриата:...,2026-08-07T14:20:30,https://news.itmo.ru/ru/education/trend/news/1...
7,14967,Студенты ИТМО заняли пять призовых мест на меж...,2026-08-05T16:14:12,https://news.itmo.ru/ru/university_live/achiev...
8,14944,"За дипломом, на сапы и в научный бар. Каким бы...",2026-07-20T16:25:07,https://news.itmo.ru/ru/university_live/leisur...


## 4. Сбор общего списка новостей

Парсер последовательно проходит страницы раздела `main_news`.

В тестовом режиме обрабатываются только первые две страницы. При полном запуске страницы обходятся до исчезновения кнопки «Следующая».

In [4]:
all_news = []

page = 1

while True:
    rows, has_next = parse_listing_page(page)

    if not rows:
        raise RuntimeError(
            f"На странице {page} не найдено ни одной новости: "
            f"{MAIN_NEWS_URL.format(page=page)}"
        )

    all_news.extend(rows)

    print(
        f"Страница {page}: "
        f"{len(rows)} новостей, "
        f"всего собрано {len(all_news)}"
    )

    if not FULL_LISTING and page >= TEST_PAGES:
        break

    if not has_next:
        break

    page += 1
    time.sleep(REQUEST_DELAY)


news_df = pd.DataFrame(
    all_news,
    columns=["id", "title", "date", "url"]
)

news_df = (
    news_df
    .drop_duplicates(subset="id")
    .sort_values("id", ascending=False)
    .reset_index(drop=True)
)

print()
print("Последняя обработанная страница:", page)
print("Итого уникальных новостей:", len(news_df))

news_df.head(10)


Страница 1: 9 новостей, всего собрано 9
Страница 2: 9 новостей, всего собрано 18
Страница 3: 9 новостей, всего собрано 27
Страница 4: 9 новостей, всего собрано 36
Страница 5: 9 новостей, всего собрано 45
Страница 6: 9 новостей, всего собрано 54
Страница 7: 9 новостей, всего собрано 63
Страница 8: 9 новостей, всего собрано 72
Страница 9: 9 новостей, всего собрано 81
Страница 10: 9 новостей, всего собрано 90
Страница 11: 9 новостей, всего собрано 99
Страница 12: 9 новостей, всего собрано 108
Страница 13: 9 новостей, всего собрано 117
Страница 14: 9 новостей, всего собрано 126
Страница 15: 9 новостей, всего собрано 135
Страница 16: 9 новостей, всего собрано 144
Страница 17: 9 новостей, всего собрано 153
Страница 18: 9 новостей, всего собрано 162
Страница 19: 9 новостей, всего собрано 171
Страница 20: 9 новостей, всего собрано 180
Страница 21: 9 новостей, всего собрано 189
Страница 22: 9 новостей, всего собрано 198
Страница 23: 9 новостей, всего собрано 207
Страница 24: 9 новостей, всего с

,id,title,date,url
0,15033,Соединили пышки и квантовую механику: как ИТМО...,2026-09-21T18:24:25,https://news.itmo.ru/ru/education/cooperation/...
1,15018,Эра сверхбыстрых вычислений: в ИТМО появился И...,2026-09-15T16:40:38,https://news.itmo.ru/ru/science/photonics/news...
2,15003,Олимпиадный год: итоги самой масштабной приемн...,2026-09-01T19:18:10,https://news.itmo.ru/ru/education/official/new...
3,15001,ITMO CONF 2026: как идеи проходят путь от лабо...,2026-08-31T18:44:11,https://news.itmo.ru/ru/education/trend/news/1...
4,14985,Всем по киберпышке! ИТМО и Яндекс Образование ...,2026-08-18T15:57:07,https://news.itmo.ru/ru/education/cooperation/...
5,14984,Российские школьники взяли четыре медали на Ме...,2026-08-17T16:39:54,https://news.itmo.ru/ru/university_live/achiev...
6,14973,В ИТМО завершили прием на бюджет бакалавриата:...,2026-08-07T14:20:30,https://news.itmo.ru/ru/education/trend/news/1...
7,14967,Студенты ИТМО заняли пять призовых мест на меж...,2026-08-05T16:14:12,https://news.itmo.ru/ru/university_live/achiev...
8,14944,"За дипломом, на сапы и в научный бар. Каким бы...",2026-07-20T16:25:07,https://news.itmo.ru/ru/university_live/leisur...
9,14916,ИТМО получит 196 миллионов рублей на развитие ...,2026-06-30T16:52:15,https://news.itmo.ru/ru/education/official/new...


In [5]:
news_df.to_csv(
    GENERAL_CSV,
    index=False,
    encoding="utf-8"
)

print("Сохранено:", GENERAL_CSV.resolve())
print("Строк:", len(news_df))


Сохранено: /content/news.csv
Строк: 1147


## 5. Парсинг страницы конкретной новости

Для каждой страницы новости собираются:

- идентификатор;
- название;
- дата публикации;
- количество просмотров;
- основной текст;
- теги.

На сайте встречается различная внутренняя разметка материалов, поэтому для текста используются несколько вариантов селекторов и резервный способ извлечения.

### Особенность исторических материалов

На части старых страниц ITMO.NEWS публичный счётчик просмотров отсутствует в HTML. Для таких публикаций поле `views` сохраняется как пропущенное значение (`NaN`), а не заменяется нулём, поскольку ноль означал бы известное количество просмотров и искажал данные.


### Вариативность HTML-разметки

В старых материалах сайта встречаются вложенные HTML-абзацы. Поэтому текст извлекается из контейнера статьи целиком, а не объединением результатов для каждого `<p>` по отдельности. Иначе содержимое вложенных элементов может несколько раз попадать в итоговый текст.

Некоторые интерактивные и медийные публикации не имеют обычного текстового тела в серверном HTML. Для таких материалов `text` сохраняется как пропущенное значение (`NaN`), а не заменяется служебными элементами страницы.


In [6]:
def extract_tags(soup):
    """Находит группу с подписью 'Теги'."""
    for group in soup.select("div.group"):
        label = group.select_one(".label")

        if label is None:
            continue

        if clean_text(label.get_text()) != "Теги":
            continue

        return [
            clean_text(link.get_text(" ", strip=True))
            for link in group.select("ul.tags a")
            if clean_text(link.get_text(" ", strip=True))
        ]

    return []


def extract_article_text(soup):
    """Извлекает основной текст с учётом вариантов верстки."""

    content = soup.select_one(
        ".article .content.js-mediator-article"
    )

    if content is None:
        content = soup.select_one(".article .post-content")

    if content is None:
        content = soup.select_one(".article")

    if content is None:
        return None

    # Удаляем служебные элементы внутри контейнера статьи.
    for tag in content.select(
        "script, style, .post-content__top-link"
    ):
        tag.decompose()

    # Текст извлекается из контейнера целиком.
    #
    # Это важно для старых материалов: в их HTML встречаются
    # вложенные теги <p>. Если извлекать каждый <p> отдельно,
    # текст дочерних элементов несколько раз попадёт в результат.
    text = clean_text(
        content.get_text(" ", strip=True)
    )

    if not text:
        return None

    # Некоторые интерактивные/медийные публикации не содержат
    # обычного текстового тела в серверном HTML.
    if text in {".", "К началу", ". К началу"}:
        return None

    return text


def parse_article(row):
    soup = get_soup(row["url"])

    title_tag = soup.select_one(".article h1") or soup.find("h1")

    title = (
        clean_text(title_tag.get_text(" ", strip=True))
        if title_tag
        else row["title"]
    )

    time_tag = soup.select_one(".news-info-wrapper time")

    date = (
        time_tag.get("datetime")
        if time_tag
        else row["date"]
    )

    views = None

    if time_tag is not None:
        views_tag = time_tag.select_one("span.icon.eye")

        if views_tag is not None:
            views_text = clean_text(
                views_tag.get_text(" ", strip=True)
            ) or ""

            digits = re.sub(r"\D", "", views_text)

            if digits:
                views = int(digits)

    text = extract_article_text(soup)
    tags = extract_tags(soup)

    return {
        "id": int(row["id"]),
        "title": title,
        "date": date,
        "views": views,
        "text": text,
        "tags": "; ".join(tags),
        "url": row["url"],
    }


## 6. Проверка парсера на одной новости

Перед массовым сбором проверяем, что все требуемые поля корректно извлекаются с одной страницы.

In [7]:
sample = parse_article(news_df.iloc[0])

print("ID:", sample["id"])
print("Название:", sample["title"])
print("Дата:", sample["date"])
print("Просмотры:", sample["views"])
print("Теги:", sample["tags"])
print()
print("Начало текста:")
print((sample["text"] or "")[:1000])


ID: 15033
Название: Соединили пышки и квантовую механику: как ИТМО и Яндекс Образование провели научный мини-фестиваль в пяти городах России
Дата: 2026-09-21T18:24:25+03:00
Просмотры: 134
Теги: Главное; Геймификация; Фестивали; Яндекс Образование

Начало текста:
Более четырех тысяч километров, пять городов и 6400 горячих пышек: рассказываем, как прошла « Киберпышечная », большое путешествие ИТМО и Яндекс Образования про науку и профессии будущего. За две с половиной недели брендированный фудтрак посетил Москву, Нижний Новгород, Казань, Екатеринбург и Санкт-Петербург, став точкой притяжения для тех, кто любит думать и вкусно поесть. Подробнее — в материале ITMO NEWS. Научный мини-фестиваль «Киберпышечная». Фото: ИТМО и Яндекс Образование «Киберпышечная» — совместный проект ИТМО и Яндекс Образования, который объединил популярную науку, современные технологии и городскую культуру. Главным драйвером проекта, угощением для гостей и частью игровой механики стала петербургская пышка. Чтобы по

## 7. Сбор содержимого всех новостей

В тестовом режиме обрабатываются только первые 10 новостей. После проверки результатов ограничение будет снято.

In [ ]:
ERRORS_CSV = OUTPUT_DIR / "errors.csv"
CHECKPOINT_EVERY = 25

if FULL_CONTENT:
    source_df = news_df.copy()

    # Если полный сбор был прерван, продолжаем с уже
    # сохраненного чекпоинта.
    if CONTENT_CSV.exists():
        existing_df = pd.read_csv(CONTENT_CSV)

        if not existing_df.empty and "id" in existing_df.columns:
            existing_df["id"] = existing_df["id"].astype(int)
        else:
            existing_df = pd.DataFrame()

    else:
        existing_df = pd.DataFrame()

    processed_ids = (
        set(existing_df["id"].tolist())
        if not existing_df.empty
        else set()
    )

    source_df = source_df[
        ~source_df["id"].isin(processed_ids)
    ].copy()

    content_df = existing_df.copy()

    print("Уже сохранено:", len(processed_ids))
    print("Осталось обработать:", len(source_df))

else:
    positions = sorted(set([
        0,
        len(news_df) // 2,
        len(news_df) - 1,
    ]))

    source_df = news_df.iloc[positions].copy()
    content_df = pd.DataFrame()

    print("Контрольные статьи:")
    display(source_df[["id", "title", "date", "url"]])


content_rows = []
errors = []


def save_checkpoint():
    global content_df, content_rows

    if content_rows:
        batch_df = pd.DataFrame(content_rows)

        content_df = pd.concat(
            [content_df, batch_df],
            ignore_index=True,
        )

        content_rows = []

    if not content_df.empty:
        content_df = (
            content_df
            .drop_duplicates(subset="id", keep="last")
            .sort_values("id", ascending=False)
            .reset_index(drop=True)
        )

        content_df.to_csv(
            CONTENT_CSV,
            index=False,
            encoding="utf-8",
        )

    if errors:
        pd.DataFrame(errors).to_csv(
            ERRORS_CSV,
            index=False,
            encoding="utf-8",
        )


for number, (_, row) in enumerate(
    tqdm(
        source_df.iterrows(),
        total=len(source_df),
        desc="Парсинг новостей",
    ),
    start=1,
):
    try:
        content_rows.append(parse_article(row))

    except Exception as error:
        errors.append({
            "id": int(row["id"]),
            "url": row["url"],
            "error": repr(error),
        })

    if number % CHECKPOINT_EVERY == 0:
        save_checkpoint()

    time.sleep(REQUEST_DELAY)


save_checkpoint()

print()
print("Всего сохранено статей:", len(content_df))
print("Ошибок этого запуска:", len(errors))

if errors:
    display(pd.DataFrame(errors))

content_df.head()


Уже сохранено: 0
Осталось обработать: 1147


Парсинг новостей:   0%|          | 0/1147 [00:00<?, ?it/s]

In [ ]:
content_df.to_csv(
    CONTENT_CSV,
    index=False,
    encoding="utf-8"
)

print("Сохранено:", CONTENT_CSV.resolve())
print("Строк:", len(content_df))


## 8. Проверка качества полученных данных

Проверяем:

- отсутствие повторяющихся идентификаторов;
- количество пропущенных значений;
- наличие текста;
- наличие просмотров;
- распределение длины текстов.

In [ ]:
print("Размер датасета:", content_df.shape)
print()

print("Дубликаты ID:")
print(content_df["id"].duplicated().sum())
print()

print("Пропуски:")
display(content_df.isna().sum().to_frame("missing"))

if not content_df.empty:
    text_lengths = content_df["text"].fillna("").str.len()

    print()
    print("Длина текста:")
    display(text_lengths.describe().to_frame("characters"))

    print()
    print("Пример результата:")
    display(
        content_df[
            ["id", "title", "date", "views", "tags"]
        ].head()
    )


## Вывод

В работе реализован двухэтапный парсинг сайта ITMO.NEWS.

На первом этапе формируется общий список публикаций с идентификатором, названием, датой и URL. На втором этапе открывается страница каждой новости и извлекаются подробные данные: количество просмотров, текст и теги.

Полученные данные сохраняются в `news.csv` и `news_content/news_content.csv`. Парсер учитывает вариативность HTML-разметки текста и использует резервные селекторы при отсутствии основной структуры.